# Minggu 4 — Encoding & scaling

**Sub-CPMK:** **P3**

Titanic (Seaborn): `ColumnTransformer` + imputer; **split dulu**, lalu `fit_transform` hanya di data latih.

**Praproses sklearn (Titanic).** Memisahkan `X` dan `y`, split stratifikasi, lalu `ColumnTransformer` dengan `Pipeline` per cabang numerik/kategorikal. `fit_transform` hanya pada `X_train`; `transform` pada `X_test` agar statistik imputasi dan scaler tidak "bocor" dari data uji.

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import pandas as pd

df = sns.load_dataset("titanic")
df = df.drop(columns=["deck"], errors="ignore")  # kolom deck terlalu banyak NA

target_col = "survived"
drop_extra = ["class", "who", "adult_male", "embark_town", "alive", "alone"]  # redundan/turunan untuk contoh ini
X = df.drop(columns=[target_col] + drop_extra, errors="ignore")
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # stratify: kelas survived seimbang di train/test
)

num_features = ["pclass", "age", "sibsp", "parch", "fare"]
cat_features = ["sex", "embarked"]

numeric_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),  # NA numerik -> median (dari train)
    ("scale", StandardScaler()),  # z-score berdasarkan statistik train
])
categorical_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("oh", OneHotEncoder(handle_unknown="ignore")),  # kategori baru di test -> vektor nol
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_features),
        ("cat", categorical_pipe, cat_features),
    ]
)

X_train_prep = preprocess.fit_transform(X_train)  # fit hanya train
X_test_prep = preprocess.transform(X_test)
print("train", X_train_prep.shape, "test", X_test_prep.shape)

**Alternatif pandas.** `get_dummies` pada `X_train` saja menghasilkan DataFrame lebar untuk eksplorasi cepat; untuk produksi tetap disarankan `Pipeline`/`ColumnTransformer` agar transform konsisten pada data baru.

In [ ]:
X_num = X_train[num_features].copy()  # subset numerik dari train
X_cat = X_train[cat_features].copy()
X_dum = pd.get_dummies(X_cat, drop_first=True)  # one-hot; drop_first mengurangi kolinearitas
X_prep_df = pd.concat([X_num.reset_index(drop=True), X_dum], axis=1)  # gabung horizontal
X_prep_df.shape  # dimensi matriks eksplorasi